
given 2 agents play each other you have to predict how much is an agent superior over another

they are playing turn based games,

Advantagep1 means the the first player agent advantage in that game( it is gathered by bringing 2 identical agents and letting them play 100 games, advantagep1 = 1 means first player win all the time so it is unfair to play first while aaround 0.5 means that there is no huge advantage to play first, 0 means player 2 always have an advantage)

you have to predict the target utility_agent1

utility_agent1- in this game. This value will be between -1 (if the first agent lost every single game) and 1 (if the first agent won every single game). Utility is calculated as (n_games_won - n_games_lost) / n_games.

both strategy_A,strategy_B are the parameters of the 2 agents, each string have 4 parameters you can split them by the (-) they all start by the prefix of "TreeBot"


Comp link: https://www.kaggle.com/t/5aaeacdf858d4a9a80a94ca18b88f022

In [1]:
# ============================================================
# 5-Fold LightGBM baseline with strategy parsing + swap augmentation
#
# Features:
#   strategy_A split into 4 parts
#   strategy_B split into 4 parts
#   advantage_p1
#
# Train augmentation:
#   swap A/B
#   advantage_p1 -> 1 - advantage_p1
#   target -> -target
#
# TTA:
#   pred_original
#   pred_swapped = -model(swapped test)
#   final = average(pred_original, pred_swapped)
#
# 5 folds:
#   train 5 models
#   average test predictions
# ============================================================

import os
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

try:
    import lightgbm as lgb
except ImportError:
    import sys
    import subprocess

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "lightgbm",
    ])

    import lightgbm as lgb


# ============================================================
# Config
# ============================================================

N_FOLDS = 5
SEED = 42


# ============================================================
# Find dataset automatically
# ============================================================

def find_dataset_root():
    for root, dirs, files in os.walk("/kaggle/input"):
        files = set(files)

        if {
            "train.csv",
            "test.csv",
            "sample_submission.csv",
        }.issubset(files):
            root_path = Path(root)

            try:
                sample = pd.read_csv(root_path / "train.csv", nrows=5)

                needed = {
                    "strategy_A",
                    "strategy_B",
                    "advantage_p1",
                }

                if needed.issubset(set(sample.columns)):
                    return root_path

            except Exception:
                pass

    raise FileNotFoundError(
        "Could not find dataset with train.csv, test.csv, sample_submission.csv "
        "and columns strategy_A, strategy_B, advantage_p1."
    )


DATA_DIR = find_dataset_root()

print("Found dataset:")
print(DATA_DIR)


# ============================================================
# Load data
# ============================================================

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

display(train.head())
display(test.head())


# ============================================================
# Target column
# ============================================================

TARGET_CANDIDATES = [
    "result_score",
    "utility_agent1",
]

TARGET = None

for col in TARGET_CANDIDATES:
    if col in train.columns:
        TARGET = col
        break

if TARGET is None:
    raise ValueError(
        f"Could not find target column. Tried: {TARGET_CANDIDATES}"
    )

print("Target:", TARGET)


# ============================================================
# Strategy parsing
# ============================================================

def parse_strategy_string(value):
    """
    Handles both formats:

        TreeBot-S2-C_low-R2-B_off
        S2-C_low-R2-B_off
        MCTS-UCB1GRAVE-0.1-MAST-false

    Returns exactly 4 parts:
        selection, exploration, playout, bounds
    """

    parts = str(value).split("-")

    if len(parts) == 5 and parts[0] in ["TreeBot", "MCTS"]:
        parts = parts[1:]

    if len(parts) < 4:
        parts = parts + ["UNK"] * (4 - len(parts))

    if len(parts) > 4:
        parts = parts[-4:]

    return parts


def add_strategy_parts(df):
    df = df.copy()

    a_parts = df["strategy_A"].apply(parse_strategy_string)
    b_parts = df["strategy_B"].apply(parse_strategy_string)

    df["A_selection"] = a_parts.apply(lambda x: x[0])
    df["A_exploration"] = a_parts.apply(lambda x: x[1])
    df["A_playout"] = a_parts.apply(lambda x: x[2])
    df["A_bounds"] = a_parts.apply(lambda x: x[3])

    df["B_selection"] = b_parts.apply(lambda x: x[0])
    df["B_exploration"] = b_parts.apply(lambda x: x[1])
    df["B_playout"] = b_parts.apply(lambda x: x[2])
    df["B_bounds"] = b_parts.apply(lambda x: x[3])

    return df


train_fe = add_strategy_parts(train)
test_fe = add_strategy_parts(test)


# ============================================================
# Features
# ============================================================

CAT_FEATURES = [
    "A_selection",
    "A_exploration",
    "A_playout",
    "A_bounds",
    "B_selection",
    "B_exploration",
    "B_playout",
    "B_bounds",
]

NUM_FEATURES = [
    "advantage_p1",
]

FEATURES = CAT_FEATURES + NUM_FEATURES


# ============================================================
# Swap augmentation helpers
# ============================================================

def make_swapped(df, has_target=True):
    out = df.copy()

    swap_pairs = [
        ("A_selection", "B_selection"),
        ("A_exploration", "B_exploration"),
        ("A_playout", "B_playout"),
        ("A_bounds", "B_bounds"),
    ]

    for a_col, b_col in swap_pairs:
        temp = out[a_col].copy()
        out[a_col] = out[b_col]
        out[b_col] = temp

    out["advantage_p1"] = 1.0 - out["advantage_p1"].astype(float)

    if has_target:
        out[TARGET] = -out[TARGET].astype(float)

    return out


def encode_categoricals(train_df, valid_df, test_df_1, test_df_2):
    train_df = train_df.copy()
    valid_df = valid_df.copy()
    test_df_1 = test_df_1.copy()
    test_df_2 = test_df_2.copy()

    for col in CAT_FEATURES:
        combined = pd.concat(
            [
                train_df[col],
                valid_df[col],
                test_df_1[col],
                test_df_2[col],
            ],
            axis=0,
        ).astype(str)

        codes, uniques = pd.factorize(combined)

        n_train = len(train_df)
        n_valid = len(valid_df)
        n_test_1 = len(test_df_1)

        train_df[col] = codes[:n_train]
        valid_df[col] = codes[n_train:n_train + n_valid]
        test_df_1[col] = codes[n_train + n_valid:n_train + n_valid + n_test_1]
        test_df_2[col] = codes[n_train + n_valid + n_test_1:]

        train_df[col] = train_df[col].astype("category")
        valid_df[col] = valid_df[col].astype("category")
        test_df_1[col] = test_df_1[col].astype("category")
        test_df_2[col] = test_df_2[col].astype("category")

    for col in NUM_FEATURES:
        train_df[col] = train_df[col].astype(float)
        valid_df[col] = valid_df[col].astype(float)
        test_df_1[col] = test_df_1[col].astype(float)
        test_df_2[col] = test_df_2[col].astype(float)

    return train_df, valid_df, test_df_1, test_df_2


# ============================================================
# Prepare test TTA versions
# ============================================================

test_original = test_fe.copy()

test_swapped = make_swapped(
    test_fe,
    has_target=False,
)


# ============================================================
# 5-Fold training
# ============================================================

kf = KFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=SEED,
)

oof_pred = np.zeros(len(train_fe))
test_pred_folds = np.zeros((len(test_fe), N_FOLDS))

fold_scores = []

for fold, (train_idx, valid_idx) in enumerate(kf.split(train_fe), start=1):
    print()
    print("=" * 60)
    print(f"Fold {fold}/{N_FOLDS}")
    print("=" * 60)

    train_fold = train_fe.iloc[train_idx].copy().reset_index(drop=True)
    valid_fold = train_fe.iloc[valid_idx].copy().reset_index(drop=True)

    train_fold_swapped = make_swapped(
        train_fold,
        has_target=True,
    )

    train_aug = pd.concat(
        [
            train_fold,
            train_fold_swapped,
        ],
        axis=0,
        ignore_index=True,
    )

    print("Train fold:", train_fold.shape)
    print("Augmented train fold:", train_aug.shape)
    print("Valid fold:", valid_fold.shape)

    train_aug_enc, valid_enc, test_original_enc, test_swapped_enc = encode_categoricals(
        train_aug,
        valid_fold,
        test_original,
        test_swapped,
    )

    X_train = train_aug_enc[FEATURES].copy()
    y_train = train_aug_enc[TARGET].copy()

    X_valid = valid_enc[FEATURES].copy()
    y_valid = valid_enc[TARGET].copy()

    X_test_original = test_original_enc[FEATURES].copy()
    X_test_swapped = test_swapped_enc[FEATURES].copy()

    model = lgb.LGBMRegressor(
        objective="regression",
        n_estimators=3000,
        learning_rate=0.02,
        num_leaves=63,
        max_depth=-1,
        min_child_samples=20,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_alpha=0.0,
        reg_lambda=1.0,
        random_state=SEED + fold,
        n_jobs=-1,
    )

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_valid, y_valid)],
        eval_metric="rmse",
        categorical_feature=CAT_FEATURES,
        callbacks=[
            lgb.early_stopping(100),
            lgb.log_evaluation(200),
        ],
    )

    valid_pred = model.predict(X_valid)
    valid_pred = np.clip(valid_pred, -1.0, 1.0)

    oof_pred[valid_idx] = valid_pred

    mse = mean_squared_error(
        y_valid,
        valid_pred,
    )

    rmse = np.sqrt(mse)
    fold_scores.append(rmse)

    print()
    print(f"Fold {fold} RMSE: {rmse:.6f}")

    pred_original = model.predict(X_test_original)
    pred_original = np.clip(pred_original, -1.0, 1.0)

    pred_swapped_perspective = model.predict(X_test_swapped)

    # Convert swapped perspective back to original perspective
    pred_swapped_back = -pred_swapped_perspective
    pred_swapped_back = np.clip(pred_swapped_back, -1.0, 1.0)

    test_pred = 0.6 * pred_original + 0.6 * pred_swapped_back
    test_pred = np.clip(test_pred, -1.0, 1.0)

    test_pred_folds[:, fold - 1] = test_pred


# ============================================================
# Overall validation
# ============================================================

oof_mse = mean_squared_error(
    train_fe[TARGET],
    oof_pred,
)

oof_rmse = np.sqrt(oof_mse)

print()
print("=" * 60)
print("CV Results")
print("=" * 60)

for i, score in enumerate(fold_scores, start=1):
    print(f"Fold {i} RMSE: {score:.6f}")

print()
print(f"Mean fold RMSE: {np.mean(fold_scores):.6f}")
print(f"Std fold RMSE:  {np.std(fold_scores):.6f}")
print(f"OOF RMSE:       {oof_rmse:.6f}")


# ============================================================
# Average test predictions
# ============================================================

final_test_pred = test_pred_folds.mean(axis=1)
final_test_pred = np.clip(final_test_pred, -1.0, 1.0)


# ============================================================
# Create submission
# ============================================================

submission = sample_submission.copy()

prediction_col = [
    c for c in submission.columns
    if c != "Id"
][0]

submission[prediction_col] = final_test_pred

submission.to_csv("submission.csv", index=False)

print()
print("Saved submission.csv")
display(submission.head())


# ============================================================
# Save OOF predictions too
# ============================================================

oof = train[["Id"]].copy() if "Id" in train.columns else pd.DataFrame({"Id": np.arange(len(train))})
oof["true"] = train_fe[TARGET].values
oof["pred"] = oof_pred
oof.to_csv("oof_predictions.csv", index=False)

print()
print("Saved oof_predictions.csv")
display(oof.head())

Found dataset:
/kaggle/input/competitions/agent-rumble
Train shape: (37500, 5)
Test shape: (12500, 4)


,Id,strategy_A,strategy_B,advantage_p1,result_score
0,0,TreeBot-S2-C_low-R2-B_off,TreeBot-S4-C_high-R2-B_off,0.80,1.000000
1,1,TreeBot-S2-C_mid-R2-B_on,TreeBot-S4-C_mid-R2-B_on,0.01,-0.600000
2,2,TreeBot-S2-C_high-R3-B_off,TreeBot-S3-C_mid-R1-B_on,0.33,-0.200000
3,3,TreeBot-S3-C_high-R2-B_off,TreeBot-S3-C_high-R2-B_on,1.00,1.000000
4,4,TreeBot-S4-C_high-R3-B_off,TreeBot-S2-C_low-R2-B_off,0.58,-0.266667


,Id,strategy_A,strategy_B,advantage_p1
0,0,TreeBot-S3-C_mid-R3-B_on,TreeBot-S1-C_mid-R1-B_off,0.49
1,1,TreeBot-S4-C_mid-R1-B_off,TreeBot-S3-C_low-R3-B_off,0.42
2,2,TreeBot-S2-C_mid-R1-B_on,TreeBot-S4-C_high-R2-B_off,0.58
3,3,TreeBot-S4-C_high-R1-B_on,TreeBot-S1-C_low-R1-B_on,0.53
4,4,TreeBot-S2-C_mid-R3-B_on,TreeBot-S3-C_low-R2-B_on,0.47


Target: result_score

Fold 1/5
Train fold: (30000, 13)
Augmented train fold: (60000, 13)
Valid fold: (7500, 13)
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.009713 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 143
[LightGBM] [Info] Number of data points in the train set: 60000, number of used features: 9
Training until validation scores don't improve for 100 rounds
[200]	valid_0's rmse: 0.524031	valid_0's l2: 0.274608
[400]	valid_0's rmse: 0.52322	valid_0's l2: 0.273759
Early stopping, best iteration is:
[323]	valid_0's rmse: 0.523161	valid_0's l2: 0.273698

Fold 1 RMSE: 0.523161

Fold 2/5
Train fold: (30000, 13)
Augmented train fold: (60000, 13)
Valid fold: (7500, 13)
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000990 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[Li

,Id,result_score
0,0,-0.082196
1,1,0.534763
2,2,-0.187174
3,3,0.371781
4,4,0.094039



Saved oof_predictions.csv


,Id,true,pred
0,0,1.000000,0.796438
1,1,-0.600000,-0.761290
2,2,-0.200000,-0.129459
3,3,1.000000,0.784579
4,4,-0.266667,0.140931
